# 第 1 天练习 —— 用 Groq（OpenAI 兼容）做多语种闲聊

## 练习目标（理念）

验证 **Groq API Key** 是否可用，并走一遍与 OpenAI SDK **几乎相同** 的 Chat Completions 调用：

- **后端**：Groq（`base_url` 指向 Groq 的 OpenAI-compatible 端点）
- **输入**：一句 Hinglish（罗马字母写的印地语风格）用户消息
- **系统约束**：按用户所用文字/脚本原样回复（Devanagari / Roman / Tamil 等），不要擅自换脚本
- **输出**：打印助手回复，确认密钥与路由都通

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| `.env` + `load_dotenv` | 从环境变量读 `GROQ_API_KEY` |
| OpenAI Python SDK | `OpenAI(api_key=..., base_url=...)` 换底座 |
| `messages` | system 定人设与脚本镜像规则，user 放问题 |
| 模型 id | `openai/gpt-oss-120b`（Groq 上的模型名字符串） |

## 怎么跑

1. 在项目根或本目录准备 `.env`，写入 `GROQ_API_KEY=...`
2. 依次运行：导入 → 加载密钥 → 创建客户端 → 发一条测试消息
3. 若第一段打印提示找不到 key，先修好环境变量再继续


In [1]:
# ========== 导入：环境变量 + OpenAI 兼容客户端 ==========

# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量（Environment Variables）
from dotenv import load_dotenv
# 从 openai 导入 OpenAI：官方 SDK；后面会把 base_url 指到 Groq
from openai import OpenAI
# 导入标准库 os：用 os.getenv 读取 GROQ_API_KEY
import os


In [ ]:
# ========== 加载密钥：确认 GROQ_API_KEY 已注入进程 ==========

# override=True：.env 里的值覆盖已有环境变量，避免旧 key 残留
load_dotenv(override=True)
# 从环境读取 Groq 密钥；名字必须是 GROQ_API_KEY（与 .env 键一致）
api_key = os.getenv('GROQ_API_KEY')

# 没有 key 就打印提示（原文混合印地语/英语，属影响行为的字符串，保持原样）
if not api_key:
    print("bhai api key nahi hai !!! oops api key not found")
else:
    # 有 key 时给一个「能用」的反馈（不打印密钥本身，避免泄露）
    print("api key kaam kr rhi hai")


In [3]:
# ========== 客户端：同一套 OpenAI SDK，底座换成 Groq ==========

# Groq 的 OpenAI 兼容 API 根路径（/openai/v1）
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

# api_key 用上一格读到的值；base_url 指向 Groq，这样 chat.completions 会打到 Groq 而不是 api.openai.com
groq = OpenAI(
    api_key=api_key,
    base_url=GROQ_BASE_URL
)


In [ ]:
# ========== 试聊：system 定「镜像用户脚本」，user 发 Hinglish ==========

# 用户消息：Hinglish（罗马字母）；保持原样，这是要发给模型的可运行字符串
message = "kesa hai bhai hindi aati hai tujhe ?"

# messages：Chat Completions 标准结构 —— 先 system 再 user
messages = [
    {"role": "system", "content": """You are a snarky humorous assistant. 
    Always reply in the same script and language that the user uses. 
    If the user writes in Devanagari Hindi, reply in Devanagari. 
    If the user writes in Roman script (Hinglish), reply in Roman script. 
    If the user writes in Tamil script, reply in Tamil script. 
    Mirror the user's script exactly — never switch scripts on your own."""},
    {"role" : "user", "content" : message}
]

# Groq 上要调用的模型 id（字符串必须与平台可用名一致，不要擅自改）
model = "openai/gpt-oss-120b"

# 发起一次非流式补全；变量名 reponse 是原作者拼写，逻辑保持不动
reponse = groq.chat.completions.create(
    model = model,
    messages=messages
)

# 取出第一条回复的文本内容并打印到输出
print(reponse.choices[0].message.content)
